# Your agent's selection policy is worth 0.0005 AUC

*By [Georgy Mamarin](https://www.kaggle.com/georgymamarin) ·
[Autonomous Agent Prediction (Beta)](https://www.kaggle.com/competitions/autonomous-agent-prediction-beta)*

**Selection is a game with a tiny prize and a real forfeit.** Here is the mechanism nobody
argues for, straight out of the harness source — one sentence on the competition page gets it
right in passing, and none of the docs your agent reads does:

In [ ]:
import warnings
warnings.filterwarnings("ignore")          # before the imports, or their warnings still print

import os, glob, re, random, zipfile, time, textwrap, platform
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
import lightgbm as lgb

SEED = 0
def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
seed_everything()

# colourblind-safe (Okabe-Ito); one colour per role for the whole notebook
C_PUB, C_OOF, C_ORACLE, C_HEDGE, C_GREY = "#0072B2", "#E69F00", "#009E73", "#CC79A7", "#8a8a8a"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})

# Kaggle renders stock dataframes a size down from body text. Style each table itself rather
# than injecting page CSS once, so the size travels with the table wherever it renders.
def show(df, **kw):
    return display(df.style
                   .format(precision=kw.pop("precision", 4))
                   .set_table_styles([{"selector": "th",
                                       "props": [("font-size", "13px")]}])
                   .set_properties(**{"font-size": "13px"}))

# The competition data mounts at /kaggle/input/<slug> or /kaggle/input/competitions/<slug>,
# so find a file we know exists instead of assuming the layout.
hits = glob.glob("/kaggle/input/**/data/train_01/train.csv", recursive=True)
for up in ("", "../", "../../"):                      # so a local checkout works too
    if hits:
        break
    hits = glob.glob(f"{up}**/data/train_01/train.csv", recursive=True)
if not hits:
    raise FileNotFoundError("competition data not found; /kaggle/input holds "
                            f"{glob.glob('/kaggle/input/*')}")
ROOT = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
DATASETS = sorted(glob.glob(f"{ROOT}/data/train_*"))

# resolve the wheel by pattern: section 10 warns it may be reissued under a new version
WHEEL = sorted(glob.glob(f"{ROOT}/wheels/kaggle_kaggle-*.whl"))[-1]

# ----------------------------------------------------------------- the figure below
from matplotlib.patches import FancyBboxPatch

ROWS = [("Your agent calls nothing",    ["pub", "pub"], "the harness fills both"),
        ("select_submission([one_id])", ["cv",  "pub"], "the harness fills what is left"),
        ("select_submission([id, id])", ["cv",  "cv"],  "nothing left to fill")]

fig, ax = plt.subplots(figsize=(7.8, 3.3))
ax.set_xlim(0, 10); ax.set_ylim(-1.15, 3.6); ax.axis("off"); ax.grid(False)
ax.text(0, 3.30, "Two submissions are scored. You choose how many of them you pick.",
        fontsize=12.5, fontweight="bold", va="center")
for i, lab in enumerate(("slot 1", "slot 2")):
    ax.text(4.35 + i * 1.55 + 0.7, 2.75, lab, ha="center", va="center", fontsize=9, color="0.45")
for r, (label, slots, note) in enumerate(ROWS):
    y = 2.2 - r * 0.95
    ax.text(0, y, label, fontsize=10.5, va="center",
            family="monospace" if "select" in label else None)
    for i, kind in enumerate(slots):
        x = 4.35 + i * 1.55
        ax.add_patch(FancyBboxPatch((x, y - 0.28), 1.4, 0.56,
                                    boxstyle="round,pad=0.02,rounding_size=0.08",
                                    facecolor=C_OOF if kind == "cv" else C_PUB, edgecolor="none"))
        ax.text(x + 0.7, y, "your pick" if kind == "cv" else "best public", ha="center",
                va="center", color="white", fontsize=9.5, fontweight="bold")
    ax.text(7.6, y, note, fontsize=9.5, va="center", color="0.35")
ax.text(0, -0.78,
        "Leaving a slot unclaimed is not an oversight: the harness fills it with your best "
        "public score.\nThe source says so; the docs your agent reads say something narrower; "
        "no published agent argues for it.", fontsize=9.8, va="center", color="0.2")
plt.tight_layout(); plt.show()

Everything your agent can win by choosing well, played with perfect hindsight, is five
ten-thousandths of AUC. What the worst policy *loses* is 0.0166 on a single dataset, thirty
times that. So this notebook is about capping the forfeit rather than chasing the prize.
Measured on all sixteen training datasets against the ground truth the competition ships in
`solution.csv`, with no LLM in the loop. My own agent never got a graded run — a 403 from the
model proxy killed every submission I made — so everything here is offline measurement. A graded
run would have returned one AUC on one hidden dataset anyway, which could not have settled a
sixteen-dataset comparison; section 10 says what that does and does not cost.

> **New to this competition?** You submit an *agent*, not predictions. It gets a Docker sandbox,
> 60 minutes, 30 submissions and $2 of tokens, and has to explore the data, train models, submit,
> read its public score, and settle on what goes to the private leaderboard. The host's
> [demo agent](https://www.kaggle.com/code/ryanholbrook/autonomous-agent-prediction-beta-demo-agent)
> covers the mechanics. This notebook is about one decision inside that hour, and section 8 is
> the one to read if you only read one.
>
> The competition closed on 6 August 2026. Everything below still runs on the data it shipped, and
> section 11 now links the write-ups that came after it.

### What you'll learn
- What every selection policy here is actually worth, priced against a ceiling, so you can
  decide how much of your agent's hour it deserves.
- What `select_submission` really does, read out of the wheel that ships with the competition
  data, and how the docs your agent actually reads describe something narrower.
- The third selection policy this opens up, which no published agent argues for.
- Whether the public half or your agent's cross-validation is the better ruler here, measured
  rather than assumed, and why my honest answer is weaker than the headline number.
- What thirty rounds of greedy public-chasing actually cost, measured against a feedback-free
  control, and what that says about the freeroll pattern.
- How that ceiling compares with the only other published estimate of the same quantity.
- The exact lines to put in your `agent.yaml` and system prompt, and the one to take out.

## Contents
1. [The format, in the parts that matter](#1)
2. [What `select_submission` actually does](#2)
3. [Sixteen samples of what your agent will meet](#3)
4. [The measurement the training data hands you](#4)
5. [Which ruler ranks candidates better](#5)
6. [Why: the mechanism](#6)
    - [Thirty rounds of chasing: what feedback actually costs](#6a)
7. [So should your agent call it?](#7)
    - [If your agent submits five times, not thirty](#7a)
8. [Selection is nearly solved. The score is somewhere else](#8)
9. [What this means for your agent](#9)
10. [Design choices & limitations](#10)
11. [Further reading](#11)

The mechanism first, because everything below depends on it. It is in the harness source, and
the documentation in front of your agent undersells it:

<a id="1"></a>
## 1. The format, in the parts that matter

Four facts drive everything below.

**Your agent submits up to 30 times and sees a score after each one.** That score comes from a
public subset of the test rows. It is feedback, and feedback invites chasing.

**Two submissions count, not one.** The budget line the harness shows your agent reads
`Selections: 2 (choose your best for final scoring)`.

**The private score is the better of the two.** Not the average, not the last one.

**The public half is 5,000 rows.** Ten thousand test rows, split down the middle. Hold that
number: for six of the sixteen training datasets it is more data than the model was trained on,
and that turns out to be the whole mechanism.

None of that is my summary of the rules. The budgets are argument defaults in the harness
runner that ships with the data:

In [ ]:
runner = open(f"{ROOT}/run_local_eval.py").read()
for flag in ("--max-submissions", "--max-selections", "--max-time-minutes", "--max-budget-usd"):
    i = runner.index(f'"{flag}"')
    default = re.search(r"default=([^,\n]+)", runner[i:i + 400]).group(1)
    print(f"{flag:22s} default {default}")

print(f"\n{len(DATASETS)} training datasets under {ROOT}")
print("metric: ROC AUC, the competition's own (run_local_eval.py --metric, default roc_auc)")
print(f"python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} "
      f"| sklearn {sklearn.__version__} | lightgbm {lgb.__version__}")
print("runtime: two slow cells, each printing its own timing: the candidate pool in section 4\n"
      "         (14-23 min across my runs) and the thirty-round chase in section 6 (~25 min)")

The whole session, and the two moments where the score is actually decided:

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(7.8, 4.4))
ax.set_xlim(0, 10); ax.set_ylim(-0.5, 9.4); ax.axis("off"); ax.grid(False)

def _b(x, y, w, h, label, col, fs=9.3, mono=False):
    ax.add_patch(FancyBboxPatch((x, y - h/2), w, h, boxstyle="round,pad=0.03,rounding_size=0.1",
                                facecolor=col, edgecolor="none"))
    ax.text(x + w/2, y, label, ha="center", va="center", color="white", fontsize=fs,
            fontweight="bold", family="monospace" if mono else None)

def _a(p1, p2, col="0.45", lw=1.3, rad=0.0):
    ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle="-|>", mutation_scale=12, color=col,
                                 lw=lw, shrinkA=4, shrinkB=4, connectionstyle=f"arc3,rad={rad}"))

ax.text(0, 9.15, "One graded session, and the two places your score is decided",
        fontsize=12.5, fontweight="bold", va="center")
_b(0.0, 8.1, 2.5, 0.55, "your agent.yaml", C_GREY)
_a((2.5, 8.1), (3.5, 8.1))
_b(3.5, 8.1, 4.6, 0.55, "Docker sandbox: 60 min, $2, no internet", C_GREY, fs=8.8)
_b(0.6, 6.5, 3.6, 0.70, "agent loop\nexplore, train, predict", C_GREY, fs=9.0)
ax.text(6.9, 6.5, "tools: run_command,\nwrite_file, edit_file,\nget_status",
        fontsize=10, color="0.45", va="center")
_a((2.6, 7.82), (2.6, 6.86))
_a((2.6, 6.15), (2.6, 5.35))
ax.text(2.45, 5.75, "submit_predictions,\nup to 30 times", fontsize=10, color="0.45",
        va="center", ha="right")
_b(0.6, 5.0, 4.0, 0.55, "scoring engine", C_GREY)
_a((1.6, 4.72), (1.6, 4.05)); _a((3.6, 4.72), (3.6, 4.05))
_b(0.0, 3.75, 3.2, 0.55, "PUBLIC half, 5,000 rows", C_PUB, fs=8.8)
_b(3.4, 3.75, 3.4, 0.55, "PRIVATE half, 5,000 rows", C_ORACLE, fs=8.8)
ax.text(6.95, 3.75, "never shown to the agent", fontsize=10, color="0.45", va="center")
# up the empty right-hand column, so it crosses neither the scoring-engine box nor the
# arrow running down to the private half
ax.add_patch(FancyArrowPatch((3.25, 3.75), (5.05, 3.75), arrowstyle="-", color=C_PUB, lw=1.6,
                             shrinkA=4, shrinkB=0))
_a((5.05, 3.75), (5.05, 6.28), col=C_PUB, lw=1.6)
ax.text(5.35, 5.15, "the public score comes back;\nthe agent iterates on it", fontsize=10,
        color=C_PUB, va="center", ha="left", fontweight="bold")
ax.plot([0, 10], [2.85, 2.85], color="0.85", lw=1)
ax.text(0, 2.45, "at the end of the session", fontsize=9.5, color="0.3",
        fontweight="bold", va="center")
_b(0.0, 1.55, 3.9, 0.55, "select_submission(ids)", C_OOF, fs=9.0, mono=True)
ax.text(4.05, 1.74, "at most 2 ids", fontsize=10, color="0.45", va="center")
ax.text(4.05, 1.36, "any slot you leave empty is filled with your best public score",
        fontsize=10, color=C_PUB, va="center", fontweight="bold")
_a((1.95, 1.27), (1.95, 0.62))
_b(0.0, 0.35, 6.6, 0.55, "the better of the two, scored on the private half", C_ORACLE, fs=9.0)
ax.text(6.8, 0.35, "= your leaderboard score", fontsize=10, color="0.35", va="center")
plt.tight_layout(); plt.show()

> **Takeaway.** Thirty submissions, two selections, sixty minutes, two dollars. This notebook is
> about the second number, which is the only one your agent can get wrong without any error
> message telling it so.

<a id="2"></a>
## 2. What `select_submission` actually does

Everything here rests on one function, so it goes first. The harness that runs your agent
ships with the competition data as a wheel, which means its behaviour is not open to
interpretation. The cell below opens the wheel and prints the code that decides what gets
graded.

In [ ]:
from matplotlib.patches import FancyBboxPatch

TRAIL = [("agent_tools.md",                 "read by you",        "there is only one", False),
         ("select_submission docstring",     "read by your agent", "there is only one", False),
         ("the composed task prompt",        "read by your agent", "claim both yourself", False),
         ("get_final_submissions",           "read by nobody",     "your best public score", True)]

fig, ax = plt.subplots(figsize=(7.8, 3.0))
ax.set_xlim(0, 10); ax.set_ylim(-0.55, 4.6); ax.axis("off"); ax.grid(False)
ax.text(0, 4.35, "Three sources point one way. The one that executes points the other.",
        fontsize=12.5, fontweight="bold", va="center")
ax.text(0, 3.72, "where it is written", fontsize=9, color="0.45")
ax.text(6.2, 3.72, "implies the unclaimed slot gets", fontsize=9, color="0.45")
for r, (art, who, says, is_code) in enumerate(TRAIL):
    y = 3.1 - r * 0.8
    ax.text(0, y + 0.11, art, fontsize=9.6, va="center", family="monospace")
    ax.text(0, y - 0.17, who, fontsize=8.4, va="center", color="0.5")
    ax.add_patch(FancyBboxPatch((6.15, y - 0.22), 3.6, 0.44,
                                boxstyle="round,pad=0.02,rounding_size=0.06",
                                facecolor=C_ORACLE if is_code else C_OOF, edgecolor="none"))
    ax.text(7.95, y, says, ha="center", va="center", color="white", fontsize=9.2, fontweight="bold")
ax.text(0, -0.28, "Only the green line is executable. The other three are what agents "
        "are built against.", fontsize=9.6, color="0.2", va="center")
plt.tight_layout(); plt.show()

Every row of that table is checkable, so here they are in order. First the code that actually decides what gets graded:

In [ ]:
print("reading", os.path.basename(WHEEL))
with zipfile.ZipFile(WHEEL) as z:
    src = z.read("kaggle_kaggle/context.py").decode()
start = src.index("    def get_final_submissions")
end = src.index("return final", start) + len("return final")
print(src[start:end])

The docstring is the part to reread: *"Always returns up to
``max_selections`` submissions"*, and the unfilled slots go to *"the best public-scoring
submissions not already selected"*.

So an agent that never calls `select_submission` at all does not fall back to one submission.
It gets **its best two public scores**, which is a complete policy, and a good one.

This matters because the tool documentation shipped alongside says something narrower:

In [ ]:
doc = open(f"{ROOT}/kaggle-kaggle-skill/resources/agent_tools.md").read()
i = doc.index("## 7. `select_submission`")
print(doc[i:i + 260])

*"the harness defaults to the best public score"*, singular, describes a fallback that returns
one submission. The code returns two. An agent author reading only this would reasonably
conclude the second slot is wasted unless they claim it, and would write `select_submission`
into their prompt to claim it.

In fairness, one piece of documentation does get it right: the competition's Overview says, in
passing, that the session score falls back *"to the submissions with the best public score"*,
plural, *"(Just like a normal competition.)"*. That sentence is easy to read past, it is not in
the docs bundled with the data, and it is not what the agent sees. The partial version — claim
one slot and the harness still fills the other — appears nowhere but the source.

Before taking my word for any of it, watch it happen. The cell below lifts
`get_final_submissions` straight out of the wheel, executes that source, and calls it on three
imaginary submissions with known public scores. Nothing here is a reimplementation: the function
being run is the one printed above.

In [ ]:
import math, textwrap

with zipfile.ZipFile(WHEEL) as z:
    ctx_src = z.read("kaggle_kaggle/context.py").decode()
i = ctx_src.index("    def get_final_submissions")
fn_src = textwrap.dedent(ctx_src[i:ctx_src.index("return final", i) + len("return final")])

class Submission:                    # the harness's own type, reduced to what the code touches
    def __init__(self, id, public_score):
        self.id, self.public_score = id, public_score

ns = {"math": math, "Submission": Submission}
exec(fn_src, ns)                     # the shipped function itself, not a copy of it
get_final_submissions = ns["get_final_submissions"]

class FakeSession:                   # three submissions; sub_2 is the best on the public half
    _submissions = {"sub_1": Submission("sub_1", 0.812),
                    "sub_2": Submission("sub_2", 0.845),
                    "sub_3": Submission("sub_3", 0.839)}
    _max_selections, _higher_is_better = 2, True
    _selected_ids = []

for selected in ([], ["sub_1"], ["sub_1", "sub_3"]):
    FakeSession._selected_ids = selected
    graded = [s.id for s in get_final_submissions(FakeSession)]
    added = [g for g in graded if g not in selected]
    print(f"agent selects {str(selected):22s} -> graded on {graded}"
          + (f"   harness added {added}" if added else "   harness added nothing"))

The middle line is the one nobody argues for. Claim a single id and the harness quietly adds `sub_2`,
your best public score, alongside it. Claim both and it adds nothing.

That would be a forgivable documentation slip if it stayed in a markdown file the author might
skip. It does not. Here is the docstring attached to the tool the *agent* calls, which lands in
the model's own context window, together with the line that decides the final score:

In [ ]:
for path, needle, n in [("kaggle_kaggle/context.py", "If never called", 68),
                        ("kaggle_kaggle/evaluation.py", "result.final_score = best_fn", 74)]:
    with zipfile.ZipFile(WHEEL) as z:
        s = z.read(path).decode()
    i = s.index(needle)
    print(f"--- {path} ---")
    print(" ".join(s[i:i + n].split()), "\n")

So the fallback is described in the singular twice, once to you and once to your agent, and the
final score is `best_fn` over what `get_final_submissions` returned: the better of the two, not
the average and not the last.

The task prompt the harness composes for every agent pushes the same way. Step 6 of the
workflow it is given:

In [ ]:
with zipfile.ZipFile(WHEEL) as z:
    ev = z.read("kaggle_kaggle/evaluation.py").decode()
j = ev.index("6. Once satisfied")
print(textwrap.fill(" ".join(ev[j:ev.index("7. As your final", j)].split()), 88))

One corroborating detail. Here is the reference agent the host ships in `sample_submission/`,
the smallest thing that works:

In [ ]:
print(open(f"{ROOT}/sample_submission/agent.yaml").read())
print("system prompt:", open(f"{ROOT}/sample_submission/prompts/system.md").read().strip())

The `tools:` list does not contain `select_submission`. The reference agent cannot call it, and
does not need to: the harness fills every slot it can from whatever was submitted.

> **Takeaway.** Not calling `select_submission` is not an omission, it is the policy "take my
> best public scores, up to the limit". Calling it with two ids replaces that policy entirely.
> Calling it with **one** id replaces half of it and leaves the rest, which is a third option
> the documentation never mentions. Sections 5 to 9 work out whether any of this is worth
> anything.

Before any of that, the two lines that keep this notebook running at all. On the Kaggle image,
reading `data/train_14/test.csv` with pandas kills the process outright: exit code -11, no
traceback, with the C engine, the python engine and chunked reads alike. The pyarrow engine
reads the same file to identical values and survives, so every read below goes through this
wrapper. If your agent loads its data the obvious way and the sandbox dies without an error
message, this is a candidate.

In [ ]:
def read_csv(path, **kw):
    try:
        return pd.read_csv(path, engine="pyarrow", **kw)   # pandas 2.3.3 segfaults on train_14
    except Exception:
        return pd.read_csv(path, **kw)

<a id="3"></a>
## 3. Sixteen samples of what your agent will meet

The competition hands you sixteen datasets "drawn from a common family of data generating
processes". Your agent will be graded on unseen members of that family, so the first job is to
learn what varies and what does not.

In [ ]:
from collections import Counter
import gc

rows = []
for d in DATASETS:
    tr = read_csv(f"{d}/train.csv")
    te = read_csv(f"{d}/test.csv", usecols=["row_id"])     # row count only
    kinds = re.findall(r"`feature_\d+`: (\w+)", open(f"{d}/DATA.md").read())
    c = Counter(kinds)
    rows.append(dict(dataset=os.path.basename(d), train_rows=len(tr), test_rows=len(te),
                     features=len(kinds), numeric=c.get("numeric", 0),
                     categorical=c.get("categorical", 0), ordinal=c.get("ordinal", 0),
                     count=c.get("count", 0),
                     positive_rate=round(tr.target.mean(), 4),
                     missing_pct=round(100 * tr.isna().mean().mean(), 1)))
    del tr, te
    gc.collect()
fam = pd.DataFrame(rows).sort_values("train_rows").reset_index(drop=True)
show(fam)
print(f"train rows span {fam.train_rows.min():,} to {fam.train_rows.max():,} "
      f"({fam.train_rows.max()/fam.train_rows.min():.0f}x)   "
      f"features {fam.features.min()} to {fam.features.max()}")
print(f"test rows: {sorted(fam.test_rows.unique())}   "
      f"positive rate: {fam.positive_rate.min():.3f} to {fam.positive_rate.max():.3f}   "
      f"datasets with missing values: {int((fam.missing_pct>0).sum())}/{len(fam)}")
print(f"training sets smaller than a 5,000-row half: {int((fam.train_rows<5000).sum())}/{len(fam)}")

Two of those columns carry the argument, so they get a picture. Training rows on the left, feature mix on the right:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.8, 4.1))
ax[0].barh(range(len(fam)), fam.train_rows, color=C_PUB, zorder=3)
ax[0].axvline(5000, color=C_ORACLE, lw=2, ls="--", zorder=4)
ax[0].text(5400, 1.0, "5,000 rows:\nthe public half", color=C_ORACLE, fontsize=10, fontweight="bold")
ax[0].set_yticks(range(len(fam))); ax[0].set_yticklabels(fam.dataset, fontsize=10)
ax[0].set_xscale("log"); ax[0].set_xlabel("training rows (log scale)")
ax[0].set_title("Train size varies 100-fold", fontsize=11, fontweight="bold")

mix = fam[["numeric", "categorical", "ordinal", "count"]].div(fam.features, axis=0)
bottom = np.zeros(len(fam))
for col, colr in zip(mix.columns, [C_PUB, C_OOF, C_ORACLE, C_HEDGE]):
    ax[1].bar(range(len(fam)), mix[col], bottom=bottom, color=colr, label=col, zorder=3)
    bottom += mix[col].values
ax[1].set_xticks(range(len(fam)))
ax[1].set_xticklabels([d.replace("train_", "") for d in fam.dataset], rotation=90, fontsize=10)
ax[1].set_xlabel("dataset (train_NN)")
ax[1].set_ylabel("share of features")
ax[1].legend(fontsize=10, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.30), frameon=False)
ax[1].set_title("Feature mix swings end to end", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

> **Takeaway.** What changes: training rows (500 to 49,432), feature count (8 to 30), and the
> mix, from all-categorical to all-numeric. What never changes: **10,000 test rows**, a target
> between 49.3% and 51.1% positive, and missing values in every one of the sixteen. An agent
> tuned to one member of this family will meet another that looks nothing like it. The constant
> that matters most is the one at that dashed line: the public half is 5,000 rows, which is
> **larger than the entire training set** for six of the sixteen.

<a id="4"></a>
## 4. The measurement the training data hands you

Each training dataset ships a `solution.csv`, and it carries more than labels:

In [ ]:
sol = read_csv(f"{DATASETS[0]}/solution.csv")
show(sol.head(3))
print(sol.Usage.value_counts().to_dict())

That `Usage` column is the competition's own public/private split, 5,000 rows each, handed to
you with the answers attached. SIDHAARTH SHREE surfaced it in the
[A-to-Z Guide](https://www.kaggle.com/code/sidhaarthshree/autonomous-agent-prediction-a-to-z-guide),
with the standard reading: the private half is locked until the end, so do not over-trust your
public rank. That is the right instinct in almost every Kaggle competition, and this notebook
is an argument that this one is the exception.

What the column makes possible is a measurement instead of an argument: for any candidate
model we can compute the public score the agent would have seen, *and* the private score it
would actually have been graded on.

So, the experiment. On each dataset, train a pool of thirty candidates, the kind of spread an
agent might produce inside its hour, one per submission it is allowed. For each candidate,
record three numbers: out-of-fold AUC on the training data, AUC on the public half, AUC on the
private half. Then ask which of the first two is the better guide to the third.

The cross-validation is 5-fold, which is more than an agent training thirty candidates in an
hour would actually afford. Its estimates here are therefore better than yours will be.

A note on wording, since two names for one thing get tiring: **out-of-fold** and **OOF** in the
code and on the axes, **cross-validation** in the prose. They mean the same number.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(7.8, 3.0))
ax.set_xlim(0, 10); ax.set_ylim(-0.75, 4.4); ax.axis("off"); ax.grid(False)
ax.text(0, 4.15, "How one candidate becomes three numbers",
        fontsize=12.5, fontweight="bold", va="center")

def _box(x, y, w, label, col, fs=9.2):
    ax.add_patch(FancyBboxPatch((x, y - 0.275), w, 0.55,
                                boxstyle="round,pad=0.02,rounding_size=0.07",
                                facecolor=col, edgecolor="none"))
    ax.text(x + w/2, y, label, ha="center", va="center", color="white",
            fontsize=fs, fontweight="bold")

def _arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=11,
                                 color="0.45", lw=1.2, shrinkA=3, shrinkB=3))

_box(0.0, 3.2, 1.7, "train.csv", C_GREY)
_box(0.0, 1.2, 1.7, "test.csv", C_GREY)
_box(2.5, 3.2, 2.1, "5-fold CV", C_GREY)
_box(5.6, 3.2, 2.3, "out-of-fold AUC", C_OOF)
_box(2.5, 1.2, 2.1, "fold-averaged\npredictions", C_GREY, fs=8.4)
_box(5.6, 1.9, 2.3, "public AUC", C_PUB)
_box(5.6, 0.5, 2.3, "private AUC", C_ORACLE)
_arrow(1.7, 3.2, 2.5, 3.2); _arrow(4.6, 3.2, 5.6, 3.2); _arrow(1.7, 1.2, 2.5, 1.2)
_arrow(4.6, 1.35, 5.6, 1.9); _arrow(4.6, 1.05, 5.6, 0.5); _arrow(1.85, 2.95, 1.85, 1.45)
ax.text(1.95, 2.2, "the trained model\nscores these rows", fontsize=8.2, color="0.45", va="center")
ax.text(4.55, 0.02, "the two arrows split those rows by the Usage column of solution.csv",
        fontsize=8.4, color="0.45", va="center", ha="center")
ax.text(8.1, 3.2, "what your agent\nwould believe", fontsize=8.8, color="0.35", va="center")
ax.text(8.1, 1.9, "what the leaderboard\nwould have shown", fontsize=8.8, color="0.35", va="center")
ax.text(8.1, 0.5, "what was actually true", fontsize=8.8, color="0.35", va="center")
ax.text(0, -0.55, "Thirty candidates, sixteen datasets, and the third number is normally "
        "invisible.", fontsize=9.6, color="0.2", va="center")
plt.tight_layout(); plt.show()

The candidates themselves. Rounds fall as leaves grow, which is both cheaper and closer to what an agent with an hour would try. What matters is that the pool spans weak to strong, not that any single member is good:

In [ ]:
# Rounds fall as leaves grow, which is both cheaper and closer to what an agent with a
# 60-minute budget would actually try. The point is a pool that spans weak to strong,
# not a pool of expensive models.
ROUNDS = {3: (600, 300, 150), 7: (600, 300, 150), 15: (600, 300, 150),
          31: (400, 250, 120), 63: (250, 150, 80), 127: (150, 100, 60)}
CANDIDATES = []
for leaves, (n02, n05, n15) in ROUNDS.items():
    for lr, n in ((0.02, n02), (0.05, n05), (0.15, n15)):
        CANDIDATES.append((f"leaves{leaves}_lr{lr}",
                           dict(num_leaves=leaves, learning_rate=lr, n_estimators=n)))
for reg in (1.0, 10.0, 50.0):
    CANDIDATES.append((f"reg{reg}", dict(num_leaves=31, learning_rate=0.05, n_estimators=200,
                                         reg_lambda=reg, min_child_samples=40)))
for sub in (0.6, 0.8):
    CANDIDATES.append((f"subsample{sub}", dict(num_leaves=63, learning_rate=0.05, n_estimators=150,
                                               subsample=sub, subsample_freq=1, colsample_bytree=sub)))
for mcs in (5, 100):
    CANDIDATES.append((f"minchild{mcs}", dict(num_leaves=31, learning_rate=0.05, n_estimators=200,
                                              min_child_samples=mcs)))
CANDIDATES += [
    ("colsample0.4", dict(num_leaves=31, learning_rate=0.05, n_estimators=200, colsample_bytree=0.4)),
    ("leaves255",    dict(num_leaves=255, learning_rate=0.05, n_estimators=80)),
    ("lr0.3",        dict(num_leaves=31, learning_rate=0.30, n_estimators=100)),
    ("stump",        dict(num_leaves=2,  learning_rate=0.10, n_estimators=400)),
    ("heavyreg",     dict(num_leaves=63, learning_rate=0.05, n_estimators=150,
                          reg_alpha=5.0, reg_lambda=100.0, min_child_samples=80)),
]
print(f"{len(CANDIDATES)} candidates, one per submission the agent is allowed")

Now the expensive part, and the only expensive part in this notebook. Each candidate is trained
with 5-fold cross-validation on every dataset, and its averaged fold predictions are scored
separately on the public rows and the private rows. That gives thirty triples per dataset:
what the agent's own validation said, what the leaderboard would have said, and what was true.

In [ ]:
def prepare(train_df, test_df):
    y = train_df.target.values
    X = train_df.drop(columns=["row_id", "target"]); Xt = test_df.drop(columns=["row_id"])
    for c in X.columns:                       # LightGBM takes categoricals natively
        if X[c].dtype == object:
            cats = pd.Categorical(pd.concat([X[c], Xt[c]]).astype(str)).categories
            X[c] = pd.Categorical(X[c].astype(str), categories=cats)
            Xt[c] = pd.Categorical(Xt[c].astype(str), categories=cats)
    return X, y, Xt

t0 = time.time(); pools = {}; kept = {}
for d in DATASETS:
    name = os.path.basename(d)
    tr = read_csv(f"{d}/train.csv"); te = read_csv(f"{d}/test.csv")
    sol = read_csv(f"{d}/solution.csv").set_index("row_id").loc[te.row_id]
    X, y, Xt = prepare(tr, te)
    is_public = (sol.Usage == "Public").values; truth = sol.target.values
    NF = 5
    skf = StratifiedKFold(NF, shuffle=True, random_state=SEED)
    rec = []; test_preds = np.zeros((len(CANDIDATES), len(Xt)), dtype=np.float32)
    for ci, (cname, params) in enumerate(CANDIDATES):
        oof = np.zeros(len(X)); tp = np.zeros(len(Xt))
        for a, b in skf.split(X, y):
            m = lgb.LGBMClassifier(random_state=SEED, verbose=-1, **params)
            m.fit(X.iloc[a], y[a])
            oof[b] = m.predict_proba(X.iloc[b])[:, 1]
            tp += m.predict_proba(Xt)[:, 1] / NF
        test_preds[ci] = tp
        rec.append(dict(candidate=cname,
                        oof=roc_auc_score(y, oof),
                        public=roc_auc_score(truth[is_public], tp[is_public]),
                        private=roc_auc_score(truth[~is_public], tp[~is_public])))
    pools[name] = pd.DataFrame(rec)
    # keep the raw test predictions: section 6 needs to redraw the public/private split
    kept[name] = dict(preds=test_preds, truth=truth)
    print(f"  {name:9s} private AUC across the pool: "
          f"{pools[name].private.min():.4f} to {pools[name].private.max():.4f}", flush=True)
print(f"\n{len(DATASETS)} pools of {len(CANDIDATES)} candidates in {(time.time()-t0)/60:.1f} min")

That is the notebook's whole data object: four hundred and eighty candidates, each with the
number its own validation reported and the number it was actually graded on. Before summarising
it, here it is.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.8, 3.6), sharey=True)
lo = min(min(p.private.min(), p.oof.min(), p.public.min()) for p in pools.values()) - 0.01
hi = max(max(p.private.max(), p.oof.max(), p.public.max()) for p in pools.values()) + 0.01
for a, col, colour, name in [(ax[0], "oof", C_OOF, "out-of-fold estimate"),
                             (ax[1], "public", C_PUB, "public-half estimate")]:
    x = np.concatenate([p[col].values for p in pools.values()])
    y = np.concatenate([p.private.values for p in pools.values()])
    a.plot([lo, hi], [lo, hi], color="0.35", lw=1, ls="--", zorder=2)
    a.scatter(x, y, s=9, color=colour, alpha=0.45, zorder=3, edgecolors="none")
    a.set_xlim(lo, hi); a.set_ylim(lo, hi); a.set_aspect("equal")
    a.set_xlabel(f"{name}\n(what the agent would believe)", fontsize=10)
    a.set_title(f"off the line by {np.abs(x - y).mean():.4f} on average",
                fontsize=11, fontweight="bold")
ax[0].set_ylabel("private AUC\n(what was actually true)", fontsize=10)
fig.suptitle("Every candidate on every dataset: 480 points, twice",
             fontsize=12.5, fontweight="bold", y=1.04)
plt.tight_layout(); plt.show()

The dashed line is where an estimate would be perfect. Both clouds hug it, which is why this
question is worth so little: any of these candidates lands close to where its estimate said it
would. Section 5 asks the sharper question the picture cannot answer, which of the two clouds
is tighter *where it matters* — at the top, among the candidates you would actually pick.

One helper, used by every comparison below. It resamples the sixteen datasets with replacement,
turning "these datasets" into "a family like these", and reports whether a difference keeps its
sign across the resamples.

In [ ]:
# 90% interval for a mean difference, resampling the sixteen datasets with replacement
def boot_ci(diff, n=20000, seed=SEED, q=5):
    diff = np.asarray(diff, dtype=float)
    rng = np.random.default_rng(seed)
    means = diff[rng.integers(0, len(diff), size=(n, len(diff)))].mean(axis=1)
    return float(np.percentile(means, q)), float(np.percentile(means, 100 - q))

def report(diff, label):
    lo, hi = boot_ci(diff)
    stable = "yes" if lo * hi > 0 else "no"
    print(f"{label:42s} {np.mean(diff):+.5f}   90% CI [{lo:+.5f}, {hi:+.5f}]   sign holds: {stable}")

> **Takeaway.** Thirty candidates on each of sixteen datasets, each scored three ways: what the
> agent's own validation said, what the leaderboard would have said, and what was true. From
> here the selection question stops being a matter of opinion.

<a id="5"></a>
## 5. Which ruler ranks candidates better

Two ways to choose, one question. For each dataset, take the candidate the public half likes
best and the candidate the out-of-fold score likes best, and compare where each one actually
lands on the private half.

In [ ]:
comp = []
for name, p in pools.items():
    comp.append(dict(
        dataset=name,
        train_rows=int(fam.set_index("dataset").loc[name, "train_rows"]),
        by_public=p.private[p.public.idxmax()],
        by_oof=p.private[p.oof.idxmax()],
        oracle=p.private.max(),
        rank_public=spearmanr(p.public, p.private).correlation,
        rank_oof=spearmanr(p.oof, p.private).correlation,
    ))
comp = pd.DataFrame(comp).sort_values("train_rows").reset_index(drop=True)
comp["edge"] = comp.by_public - comp.by_oof
show(comp, precision=4)
print(f"mean private AUC picking by public : {comp.by_public.mean():.5f}")
print(f"mean private AUC picking by OOF    : {comp.by_oof.mean():.5f}")
print(f"public wins {int((comp.edge>0).sum())} datasets, ties {int((comp.edge==0).sum())}, "
      f"loses {int((comp.edge<0).sum())}\n")
report(comp.edge.values, "one slot: public minus out-of-fold")
lo95, hi95 = boot_ci(comp.edge.values, q=2.5)
print(f"{'the same difference at 95%':42s} {comp.edge.mean():+.5f}   95% CI "
      f"[{lo95:+.5f}, {hi95:+.5f}]   sign holds: {'yes' if lo95 * hi95 > 0 else 'no'}")

An interval is not enough on its own. A mean over sixteen can be sign-stable under resampling
and still be one or two datasets wearing a trench coat, so take each dataset out and look
again.

In [ ]:
e = comp.set_index("dataset").edge
loo = pd.DataFrame({"edge_without_it": [e.drop(d).mean() for d in e.index],
                    "share_of_total": (e / e.sum()).values}, index=e.index)
show(loo.sort_values("share_of_total", ascending=False).head(4), precision=5)
print(f"mean edge over all {len(e)}      : {e.mean():+.5f}")
print(f"median dataset                : {e.median():+.5f}")
print(f"drop the single largest contributor ({e.idxmax()}): {e.drop(e.idxmax()).mean():+.5f}")
print(f"drop the two largest                       : "
      f"{e.drop(e.abs().sort_values().index[-2:]).mean():+.5f}")
print(f"\ncount of datasets where public wins: {int((e>1e-6).sum())}, "
      f"loses: {int((e<-1e-6).sum())}, level: {int((e.abs()<=1e-6).sum())}")

The same sixteen numbers, sorted by training rows, because that ordering turns out to be the story:

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 3.6))
x = np.arange(len(comp))
ax.bar(x, comp.edge, color=[C_PUB if e >= 0 else C_OOF for e in comp.edge], zorder=3)
ax.axhline(0, color="0.3", lw=1)
ax.set_xticks(x)
ax.set_xticklabels([d.replace("train_", "") for d in comp.dataset], fontsize=10)
ax.set_xlabel("dataset (train_NN), sorted by training rows")
ax.set_ylabel("private AUC:\npublic pick minus OOF pick")
ax.set_title("When the public half wins it wins on the small datasets.\n"
             "When it loses it loses by a hair. (x-axis sorted by training rows)",
             fontsize=11, fontweight="bold")
ax.text(0.98, 0.95, f"mean {comp.edge.mean():+.5f}", transform=ax.transAxes,
        fontsize=11, fontweight="bold", va="top", ha="right", color=C_PUB)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=C_PUB, label="public pick better"),
                   Patch(facecolor=C_OOF, label="out-of-fold pick better")],
          fontsize=10, loc="upper center", bbox_to_anchor=(0.5, -0.30), ncol=2, frameon=False)
plt.tight_layout(); plt.show()

> **Takeaway, with the hedges that belong in it.** The public half picks the better candidate by
> +0.00218 AUC on average, and the sign survives the dataset bootstrap. Then read the rest of
> the output. The **median dataset shows exactly zero difference**, the win-loss count is 7 to 5
> which is a coin flip, and dropping the single largest contributor roughly halves the mean.
> This is not "the leaderboard is reliable and your validation is not". It is a handful of small
> datasets where cross-validation goes badly wrong, and a majority where the choice does not
> matter at all.
>
> Two more things to weigh against the headline. That interval is a 90% one and its lower bound
> sits at +0.00017, which is eight percent of the point estimate; the 95% interval printed beside
> it runs from −0.00014 and includes zero. And before settling on the candidate pool in section 4 I ran a different pool of thirty
> LightGBM variants, which gave the same direction at **+0.0004**, with an interval that crossed
> zero.
>
> Notice where that lands: +0.0004 is what this pool also gives once its two biggest
> contributors are removed. The two runs agree better than their headline numbers suggest, and
> they agree on the small number: **on a typical dataset from this family, which ruler you pick
> is worth about four ten-thousandths of AUC, and often exactly nothing.** Take the direction as
> the finding and the magnitude as pool-dependent.

<a id="6"></a>
## 6. Why: the mechanism

Every number so far rests on one particular assignment of ten thousand test rows into a public
half and a private half. That assignment is a coin toss the competition performed once, and the
dataset bootstrap in section 5 does not touch it: it reshuffles which *datasets* I have while
holding each dataset's split fixed.

The split is the more relevant lottery, because your agent will meet a new dataset with a new
one. And it is cheap to interrogate, since the candidates' test predictions are already
computed: redraw the 5,000/5,000 assignment a few hundred times and watch what the answer does.

In [ ]:
from scipy.stats import rankdata

def auc_rows(P, y):                       # AUC of every candidate at once, on a row subset
    r = rankdata(P, axis=1)
    npos = y.sum(); nneg = len(y) - npos
    return (r[:, y == 1].sum(axis=1) - npos * (npos + 1) / 2) / (npos * nneg)

REDRAWS = 300
split_rows = []
for di, (name, k) in enumerate(kept.items()):
    P, y = k["preds"], k["truth"]
    oof_best = pools[name].oof.values.argmax()
    rng = np.random.default_rng(500 + di)
    e = np.empty(REDRAWS)
    for r in range(REDRAWS):
        idx = rng.permutation(len(y)); pub, prv = idx[:len(y)//2], idx[len(y)//2:]
        a_pub = auc_rows(P[:, pub], y[pub]); a_prv = auc_rows(P[:, prv], y[prv])
        e[r] = a_prv[a_pub.argmax()] - a_prv[oof_best]
    split_rows.append(dict(dataset=name,
                           train_rows=int(fam.set_index("dataset").loc[name, "train_rows"]),
                           mean_edge=e.mean(), lo=np.percentile(e, 5), hi=np.percentile(e, 95),
                           pct_public_wins=100 * (e > 1e-6).mean()))
split = pd.DataFrame(split_rows).sort_values("train_rows").reset_index(drop=True)
show(split, precision=5)
pub_side = split[split.pct_public_wins > 90]
oof_side = split[split.pct_public_wins < 10]
print(f"public reliably better (wins >90% of redraws): {len(pub_side)} datasets, "
      f"average effect {pub_side.mean_edge.mean():+.5f}")
print(f"out-of-fold reliably better (<10%)           : {len(oof_side)} datasets, "
      f"average effect {oof_side.mean_edge.mean():+.5f}")
print(f"so cross-validation wins more often, and the public half wins "
      f"{abs(pub_side.mean_edge.mean()/oof_side.mean_edge.mean()):.0f}x bigger")

Each bar below is one dataset's verdict across three hundred redrawn splits. Narrow bars mean the split lottery is not what decides the answer:

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 3.6))
y = np.arange(len(split))
colors = [C_PUB if p > 90 else C_OOF if p < 10 else C_GREY for p in split.pct_public_wins]
ax.hlines(y, split.lo, split.hi, color=colors, lw=4, alpha=0.75, zorder=2)
ax.scatter(split.mean_edge, y, s=30, color="0.15", zorder=3)
ax.axvline(0, color="0.3", lw=1)
ax.set_yticks(y)
ax.set_yticklabels([d.replace("train_", "") for d in split.dataset], fontsize=10)
ax.set_ylabel("dataset (train_NN)")
ax.set_xlabel("public pick minus out-of-fold pick, private AUC\n(bar = 90% of 300 redrawn splits)")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=C_PUB, label="public wins >90% of redraws"),
                   Patch(facecolor=C_OOF, label="out-of-fold does"),
                   Patch(facecolor=C_GREY, label="neither")],
          fontsize=10, loc="lower right", frameon=False)
ax.set_title("Redrawing the split barely moves any dataset's verdict.\n"
             "The verdicts just disagree with each other.", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

That table dismantles the tidy rule I was about to give you.

The split lottery is **not** where the uncertainty lives. Eleven of the sixteen datasets return
the same winner in at least nine redraws out of ten, and the four with the largest effects
return it in essentially all three hundred. Section 5's +0.00218 was not a lucky coin toss.
The five that sit on the fence are the ones where the two rulers are level anyway.

But the verdicts disagree, and the shape of the disagreement is the finding.
**Cross-validation is the reliable winner on more datasets than the public half is, seven to
four. The public half's wins are about fourteen times larger.** Its four are led by the two
smallest datasets in the set, train_13 at +0.0165 and train_15 at +0.0080, with train_09 at
+0.0072 close behind. Cross-validation's seven are worth around six ten-thousandths each.

That also breaks the tidy rule. train_05 has 1,060 training rows, sits well inside my "under
5,000" group, and goes to cross-validation in every single redraw.

So the honest mechanism is about **magnitude, not frequency**. Pick by cross-validation and you
will be right slightly more often and wrong expensively on the small datasets; pick by the public
half and you lose a hair more often and never lose much. That is a weaker claim than "trust the
leaderboard", it is the one the evidence supports, and it is the same asymmetry that section 7
finds in the policies themselves.

Rank correlation shows the same thing from another angle.

In [ ]:
small = comp[comp.train_rows < 5000]; large = comp[comp.train_rows >= 5000]
show(pd.DataFrame([
    dict(group=f"train < 5,000 rows  (n={len(small)})", rank_public=small.rank_public.mean(),
         rank_oof=small.rank_oof.mean(), public_edge=small.edge.mean()),
    dict(group=f"train >= 5,000 rows (n={len(large)})", rank_public=large.rank_public.mean(),
         rank_oof=large.rank_oof.mean(), public_edge=large.edge.mean()),
    dict(group=f"all {len(comp)} together", rank_public=comp.rank_public.mean(),
         rank_oof=comp.rank_oof.mean(), public_edge=comp.edge.mean()),
]), precision=4)
worst = comp.loc[comp.rank_oof.idxmin()]
print(f"worst case for cross-validation: {worst.dataset} with {worst.train_rows:,} training rows. "
      f"OOF orders the pool at Spearman {worst.rank_oof:+.3f} against private, "
      f"the public half at {worst.rank_public:+.3f}")

The same question asked by rank correlation, with the two rulers joined per dataset so you can read the gap directly:

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 3.4))
ax.scatter(comp.train_rows, comp.rank_public, s=55, color=C_PUB, label="public half", zorder=3)
ax.scatter(comp.train_rows, comp.rank_oof, s=55, color=C_OOF, marker="s",
           label="out-of-fold", zorder=3)
for _, r in comp.iterrows():
    ax.plot([r.train_rows, r.train_rows], [r.rank_oof, r.rank_public], color=C_GREY, lw=1, zorder=2)
ax.axvline(5000, color=C_ORACLE, lw=2, ls="--", zorder=1)
ax.set_xscale("log")
ax.set_xlabel("training rows (log scale)\n"
              "dashed line = 5,000 rows: left of it the training set is smaller than the public half")
ax.set_ylabel("Spearman vs private AUC")
ax.set_ylim(0.45, 1.03)          # keep train_09's +0.541 off the axis edge
ax.legend(fontsize=9, loc="lower right")
ax.set_title("Left of the line the public half is reliably the better ruler.\n"
             "Right of it, the two trade places.",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

The split in that table is the whole mechanism, and it is sharper than I expected.

On the six datasets with fewer than 5,000 training rows, the public half orders the pool at
Spearman 0.949 against 0.813 for cross-validation, and picking on it is worth +0.0061. On the
ten larger ones the two rulers are level, 0.919 against 0.898, and the public half is worth
**-0.0002**, which is to say nothing at all.

So the honest statement is not "the leaderboard beats your validation". It is narrower: **the
entire advantage lives in the datasets where 5,000 public rows are more data than the agent
trained on.** An AUC estimated from 500 rows is a noisy quantity; an AUC on 5,000 rows is not,
and no amount of folding fixes the first problem. Six of these sixteen datasets are in that
regime, and `len(train)` tells your agent which one it landed in before it trains anything.

Rank correlation says which ruler *orders* better. This asks a blunter question: how far does
each ruler's score sit from the private score it stands in for, once the constant offset a
ruler is allowed to have is removed?

One caveat to read the numbers with. Both columns are differences against the private half, so
both carry the private half's own sampling noise. That inflates each column and pushes their
*ratio* toward 1, which means the ratios below understate the gap rather than flatter it.

In [ ]:
err = pd.DataFrame([dict(dataset=n, train_rows=int(fam.set_index("dataset").loc[n, "train_rows"]),
                         public=(p.public - p.private).std(),
                         oof=(p.oof - p.private).std()) for n, p in pools.items()]).set_index("dataset")
err["ratio"] = err.oof / err.public
show(err.sort_values("train_rows"), precision=5)
print(f"spread between the two test halves       : {err.public.mean():.5f}")
print(f"spread between out-of-fold and private   : {err.oof.mean():.5f}")
print(f"out-of-fold is the wider of the two on {int((err.ratio>1).sum())} of {len(err)} datasets\n")
for label, sub in [("train < 5,000", err[err.train_rows < 5000]),
                   ("train >= 5,000", err[err.train_rows >= 5000])]:
    print(f"  median ratio, {label:15s}: {sub.ratio.median():.1f}x")

The noise table splits the same way. The
out-of-fold ruler sits **3.1x** further from the private half than the public ruler does on the
small datasets, and **1.4x** on the large ones. Most of the gap is sample size, exactly as the
first reason says. But it does not close to 1.0 even at 49,432 training rows, and given the
attenuation noted above, the real residual is wider than 1.4x.

That residual is the second reason, and it is structural rather than statistical: the public
and private halves are two halves of one 10,000-row draw, so whatever makes that test set what
it is, both halves share it. The training rows are a different sample entirely. This effect is
small, it never disappears, and it is why I would still lean public at parity rather than
calling a tie.

> **Takeaway.** Prefer the public half when your training set is small relative to 5,000 rows.
> When it is not, the two rulers are close enough that the choice stops earning attention, and
> section 8 argues it never earned much. Out-of-fold is the wider ruler on 15 of 16 datasets,
> but the size of that gap tracks training rows, which is a sample-size fact before it is
> anything more interesting.

<a id="6a"></a>
### Thirty rounds of chasing: what feedback actually costs

Everything above treats the candidates as fixed: train thirty things, then choose. A real
agent does something more dangerous. It reads the public score after every submission and
*adapts*: keeps the tweak when the number goes up, reverts when it goes down. Thirty adaptive
queries against the same 5,000 rows is the textbook recipe for overfitting a leaderboard, and
the freeroll strategies now circulating spend their whole leftover budget doing exactly this.

It is also measurable. Below, a greedy chaser starts from a mid-grid LightGBM and spends the
entire submission budget: each round it proposes one random tweak, a hyperparameter nudge or
dropping up to three random features, trains on the full training set, and keeps the tweak
only if the public half improves. The control spends the same thirty fits with no feedback at
all: independent proposals around the same base, best public taken once at the end. Whatever
optimism the chaser accumulates *beyond* the control is the price of adaptivity itself.

In [ ]:
CHASE_BASE = dict(num_leaves=31, learning_rate=0.05, n_estimators=200)
T_CHASE = 30

def propose(cfg, feats, all_feats, rng):
    # one random tweak: nudge a hyperparameter, or re-draw which features are dropped
    cfg = dict(cfg); feats = list(feats)
    knob = rng.choice(["num_leaves", "learning_rate", "n_estimators",
                       "min_child_samples", "reg_lambda", "colsample_bytree", "features"])
    if knob == "num_leaves":
        cfg["num_leaves"] = int(np.clip(cfg.get("num_leaves", 31) * rng.choice([0.5, 2/3, 1.5, 2]), 2, 255))
    elif knob == "learning_rate":
        cfg["learning_rate"] = float(np.clip(cfg.get("learning_rate", .05) * rng.choice([0.5, 0.7, 1.4, 2]), .005, .5))
    elif knob == "n_estimators":
        cfg["n_estimators"] = int(np.clip(cfg.get("n_estimators", 200) * rng.choice([0.6, 0.8, 1.25, 1.5]), 50, 400))
    elif knob == "min_child_samples":
        cfg["min_child_samples"] = int(rng.choice([5, 10, 20, 40, 80]))
    elif knob == "reg_lambda":
        cfg["reg_lambda"] = float(rng.choice([0.0, 1.0, 10.0, 50.0]))
    elif knob == "colsample_bytree":
        cfg["colsample_bytree"] = float(rng.choice([0.4, 0.6, 0.8, 1.0]))
    else:
        drop = rng.integers(0, min(3, len(all_feats) - 2) + 1)
        feats = list(rng.choice(all_feats, size=len(all_feats) - drop, replace=False))
    return cfg, feats

That is the whole treatment: one knob or one feature subset per round, chosen at random. Now the
two runs it feeds, the greedy chain and its feedback-free control.

In [ ]:
t0 = time.time(); rows_chase = []
for d in DATASETS:
    name = os.path.basename(d)
    tr = read_csv(f"{d}/train.csv"); te = read_csv(f"{d}/test.csv")
    sol = read_csv(f"{d}/solution.csv").set_index("row_id").loc[te.row_id]
    y = tr.target.values
    X, _, Xt = prepare(tr, te)
    is_pub = (sol.Usage == "Public").values; truth = sol.target.values
    all_feats = list(X.columns)

    def fit_score(cfg, feats):
        m = lgb.LGBMClassifier(random_state=SEED, verbose=-1, **cfg)
        m.fit(X[feats], y)
        p = m.predict_proba(Xt[feats])[:, 1]
        return roc_auc_score(truth[is_pub], p[is_pub]), roc_auc_score(truth[~is_pub], p[~is_pub])

    base_pub, base_prv = fit_score(CHASE_BASE, all_feats)

    rng = np.random.default_rng(42)          # the greedy chain: accept iff public improves
    cur_cfg, cur_feats, cur_pub, cur_prv = dict(CHASE_BASE), list(all_feats), base_pub, base_prv
    accepts = 0
    for _ in range(T_CHASE):
        cfg, feats = propose(cur_cfg, cur_feats, all_feats, rng)
        pub, prv = fit_score(cfg, feats)
        if pub > cur_pub:
            cur_cfg, cur_feats, cur_pub, cur_prv, accepts = cfg, feats, pub, prv, accepts + 1

    rng2 = np.random.default_rng(43)         # the control: same budget, no feedback
    best_pub, best_prv = base_pub, base_prv
    for _ in range(T_CHASE):
        cfg, feats = propose(CHASE_BASE, all_feats, all_feats, rng2)
        pub, prv = fit_score(cfg, feats)
        if pub > best_pub:
            best_pub, best_prv = pub, prv

    rows_chase.append(dict(dataset=name, train_rows=len(X),
                           base_pub=base_pub, base_prv=base_prv,
                           chase_pub=cur_pub, chase_prv=cur_prv, accepts=accepts,
                           indep_pub=best_pub, indep_prv=best_prv))
    print(f"  {name}: accepted {accepts:2d}/30, private {base_prv:.4f} -> {cur_prv:.4f} "
          f"(control {best_prv:.4f})", flush=True)

chase_df = pd.DataFrame(rows_chase)
print(f"\n16 chases + 16 controls in {(time.time()-t0)/60:.1f} min")
for lab, pubc, prvc in [("base", "base_pub", "base_prv"), ("chase", "chase_pub", "chase_prv"),
                        ("control", "indep_pub", "indep_prv")]:
    g = (chase_df[pubc] - chase_df[prvc]).mean()
    print(f"mean public-minus-private gap, {lab:8s}: {g:+.5f}")
base_gap = chase_df.base_pub - chase_df.base_prv
print(f"\noptimism GAINED over the base:   chase   "
      f"{((chase_df.chase_pub-chase_df.chase_prv) - base_gap).mean():+.5f}")
print(f"                                 control "
      f"{((chase_df.indep_pub-chase_df.indep_prv) - base_gap).mean():+.5f}")
print(f"\nmean private gain over the base: chase   {(chase_df.chase_prv-chase_df.base_prv).mean():+.5f}")
print(f"                                 control {(chase_df.indep_prv-chase_df.base_prv).mean():+.5f}")
report((chase_df.chase_prv - chase_df.indep_prv).values, "chase minus control, private")

Both effects, dataset by dataset. The left panel is the one the scare story never mentions:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7.8, 3.8))
o = chase_df.sort_values("train_rows").reset_index(drop=True)
yy = np.arange(len(o)); h = 0.38
ax[0].barh(yy + h/2, o.chase_prv - o.base_prv, h, color=C_PUB, zorder=3, label="greedy chase")
ax[0].barh(yy - h/2, o.indep_prv - o.base_prv, h, color=C_GREY, zorder=3, label="no-feedback control")
ax[0].axvline(0, color="0.3", lw=1)
ax[0].set_yticks(yy)
ax[0].set_yticklabels([d.replace("train_", "") for d in o.dataset], fontsize=10)
ax[0].set_ylabel("dataset (train_NN), sorted by training rows")
ax[0].set_xlabel("private AUC gain over the base")
ax[0].legend(fontsize=10, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2,
             frameon=False)
ax[0].set_title("Chasing the public half\nbeat blind search", fontsize=11, fontweight="bold")
infl_chase = (o.chase_pub - o.chase_prv) - (o.base_pub - o.base_prv)
infl_indep = (o.indep_pub - o.indep_prv) - (o.base_pub - o.base_prv)
ax[1].scatter(np.zeros(len(o)) + 0, infl_chase, s=40, color=C_PUB, alpha=0.7, zorder=3)
ax[1].scatter(np.zeros(len(o)) + 1, infl_indep, s=40, color=C_GREY, alpha=0.7, zorder=3)
for i in range(len(o)):
    ax[1].plot([0, 1], [infl_chase[i], infl_indep[i]], color="0.85", lw=0.8, zorder=2)
ax[1].scatter([0, 1], [infl_chase.mean(), infl_indep.mean()], s=180, marker="_",
              color="0.1", zorder=4)
ax[1].set_xticks([0, 1])
ax[1].set_xticklabels(["greedy\nchase", "no-feedback\ncontrol"], fontsize=10)
ax[1].set_xlim(-0.5, 1.5)
ax[1].set_ylabel("public-score optimism gained\n(gap change vs the base)")
ax[1].set_title("And bought the same tiny\noptimism as not chasing", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

> **Takeaway.** The scare story does not survive the measurement. Thirty greedy rounds
> inflated the public read by **+0.0021** on average; the feedback-free control inflated it
> by **+0.0023**. Adaptivity added nothing beyond the ordinary best-of-thirty selection bias,
> because thirty queries is far too small a budget to mine 5,000 rows. Meanwhile the chaser's
> private score improved by **+0.0136** against **+0.0060** for blind search with the same
> thirty fits: section 5's conclusion cuts both ways, and a ruler this good is safe to
> optimize against this hard. That is the measured case for the freeroll pattern of banking a
> baseline and grinding against the public board, and it is one greedy LightGBM chain per
> dataset, so read it as the cost of *this* kind of chasing, not of every kind.

<a id="7"></a>
## 7. So should your agent call it?

Four policies, scored the way the harness scores them: the better of the two selections on the
private half. Two of them are what published agent prompts actually produce. `override_both` is
the natural misreading of "select your best two submissions" rather than a prompt I found anyone
writing, and `one_slot_only` is a bound rather than a competitor.

- **`harness_default`** is what happens when the agent never calls the tool: its two best
  public scores.
- **`override_both`** is `select_submission([cv_1, cv_2])`, the agent claiming both slots with
  its own cross-validation favourites. This is what "lock in your top 2" produces.
- **`override_one`** is `select_submission([cv_1])`, one id only. Look again at the source in
  section 2: `remaining_slots = self._max_selections - len(final)`, so the harness still fills
  the second slot with the best public score the agent did not claim. This is what the common
  instruction "call `select_submission` with your best submission ID" produces, and it is a
  gentler move than it looks.
- **`one_slot_only`** is the counterfactual where the format allowed a single selection. It is
  a bound rather than a competitor, since the better of two can never be worse than the first
  of them, and it is here only to price the second slot.

One property to keep in mind while reading the table: `override_both` claims the agent's top
*two* cross-validation picks while `override_one` claims only the first, so wherever the
agent's second pick happens to be the winner, `override_both` gets it and `override_one` does
not. Its upside is therefore never smaller by construction. Watch whether that ever cashes in.

In [ ]:
two = []
for name, p in pools.items():
    prv = p.private.values
    pub_order = p.public.values.argsort()[::-1]
    oof_order = p.oof.values.argsort()[::-1]
    # agent claims one slot with its CV favourite; the harness fills the other with the
    # best public score it did not claim
    filled = [i for i in pub_order if i != oof_order[0]][0]
    two.append(dict(dataset=name,
                    agree=int(pub_order[0] == oof_order[0]),
                    harness_default=prv[pub_order[:2]].max(),
                    override_both=prv[oof_order[:2]].max(),
                    override_one=prv[[oof_order[0], filled]].max(),
                    one_slot_only=prv[pub_order[0]],
                    oracle=prv.max()))
two = pd.DataFrame(two).set_index("dataset")
show(two, precision=4)
print(two[["harness_default", "override_both", "override_one", "one_slot_only", "oracle"]]
        .mean().sort_values(ascending=False).round(5).to_string())
print(f"\nthe two rulers already agree on the top candidate in "
      f"{int(two.agree.sum())} of {len(two)} datasets\n")
report((two.override_both - two.harness_default).values, "override both slots with CV picks")
report((two.override_one  - two.harness_default).values, "override one slot with a CV pick")
report((two.oracle        - two.harness_default).values, "what perfect hindsight would add")

The means are close enough that comparing them alone would be a mistake, and two of the three
intervals cross zero. Before looking at the differences, here is the part that does not depend
on any of these sixteen datasets.

Claiming **one** slot keeps the best public submission in the other one, always. So the most
this policy can cost you, against doing nothing, is however much the *second*-best public
submission would have beaten the best one by on the private half. That is a bounded quantity,
and on this pool it is small.

Claiming **both** slots keeps nothing. Its cost is bounded only by how far the agent's two
cross-validation picks can fall below the two public ones, which has no small bound at all.

The arithmetic below is an illustration of that asymmetry, not the reason for it.

In [ ]:
delta = pd.DataFrame({"override_both": two.override_both - two.harness_default,
                      "override_one":  two.override_one  - two.harness_default})
for c in delta:
    v = delta[c]
    print(f"{c:14s} mean {v.mean():+.5f}   best case {v.max():+.5f}   worst case {v.min():+.5f} "
          f"({v.idxmin()})   wins {int((v>0).sum())} ties {int((v==0).sum())} losses {int((v<0).sum())}")
report((two.override_one - two.override_both).values, "one id minus two ids")
policies = two[["harness_default", "override_both", "override_one", "one_slot_only"]]
spread = policies.max(axis=1) - policies.min(axis=1)
for tol in (0.001, 0.0025):
    print(f"datasets where all four policies sit within {tol:.4f} of each other: "
          f"{int((spread <= tol).sum())} of {len(spread)}")
worst = delta.override_both.idxmin()
rest = delta.drop(index=[worst, delta.override_both.nsmallest(2).index[-1]])
print(f"\nthe two policies differ by {(delta.override_one - delta.override_both).mean():+.5f} on average,")
print(f"but drop the two datasets that drive it and passing two is {rest.override_both.mean():+.5f}, "
      f"which is to say level")

Both overrides, dataset by dataset, against doing nothing. The empty middle is the finding:

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 3.5))
o = delta.reindex(delta.override_both.sort_values().index)
x = np.arange(len(o)); w = 0.4
ax.bar(x - w/2, o.override_both, w, color=C_OOF, zorder=3, label="pass two ids")
ax.bar(x + w/2, o.override_one,  w, color=C_HEDGE, zorder=3, label="pass one id")
ax.axhline(0, color="0.3", lw=1)
ax.set_xticks(x)
ax.set_xticklabels([d.replace("train_", "") for d in o.index], fontsize=10)
ax.set_xlabel("dataset (train_NN)")
ax.set_ylabel("private AUC vs\ncalling nothing")
ax.legend(fontsize=9, loc="lower right")
ax.set_title("Both overrides win the same three datasets.\n"
             "Only one of them gives up 0.017 of AUC on train_13.",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

> **Takeaway, and it is a modest one.** On thirteen of the sixteen datasets every policy in this
> section lands within 0.0025 AUC of every other, and on eight of them within 0.001 — both counts
> printed above. Selection is mostly a non-event, and any
> version of this notebook that told you otherwise would be selling something.
>
> Where they differ, they differ in the direction the bounded-versus-unbounded argument above
> predicts. Passing one id was never worse than 0.00098 against doing nothing. Passing two lost
> **0.01664** on train_13, the 500-row dataset, exactly where section 6 says cross-validation is
> least trustworthy and where an agent has the least chance of noticing.
>
> Three qualifications. Passing two never cashed in its extra upside: both overrides top out at
> the same +0.00221. Head to head one id beats two by +0.00172 and that interval clears zero,
> but against the *default* neither override is sign-stable, so I am not claiming passing one id
> gains you anything. And that +0.00172 rests on two datasets: remove train_13 and its nearest
> neighbour and passing two stops being negative at all. It is a rare large loss, not a robust
> average effect.
>
> Which is the argument, and it does not need an average: passing one id costs nothing
> measurable and caps a downside that passing two leaves open. When two options are
> indistinguishable in expectation, take the one with the shorter tail.

<a id="7a"></a>
### If your agent submits five times, not thirty

The pool has thirty candidates because thirty is the submission cap, and most agents stop
well short of it. Subsampling the pool prices every policy at the budget your agent actually
uses: for each K, draw K of the thirty candidates at random, three hundred times, and score
the same policies inside each draw.

In [ ]:
rng_rows = []
for K in (5, 10, 15, 20, 30):
    per = dict(headroom=[], d_both=[], d_one=[])
    for name, p in pools.items():
        oof_v, pub_v, prv_v = p.oof.values, p.public.values, p.private.values
        rng = np.random.default_rng(1000 + K)
        hb, db, do = [], [], []
        for _ in range(300 if K < len(prv_v) else 1):
            idx = rng.choice(len(prv_v), size=K, replace=False) if K < len(prv_v) else np.arange(len(prv_v))
            o, u, v = oof_v[idx], pub_v[idx], prv_v[idx]
            pub_o = u.argsort()[::-1]; oof_o = o.argsort()[::-1]
            default = v[pub_o[:2]].max()
            filled = [i for i in pub_o if i != oof_o[0]][0]
            hb.append(v.max() - default)
            db.append(v[oof_o[:2]].max() - default)
            do.append(max(v[oof_o[0]], v[filled]) - default)
        per["headroom"].append(np.mean(hb)); per["d_both"].append(np.mean(db)); per["d_one"].append(np.mean(do))
    rng_rows.append(dict(K=K, headroom=np.mean(per["headroom"]),
                         override_both=np.mean(per["d_both"]), override_one=np.mean(per["d_one"])))
kcurve = pd.DataFrame(rng_rows)
show(kcurve, precision=5)

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 3.2))
ax.plot(kcurve.K, kcurve.headroom, "-o", color=C_ORACLE, label="hindsight ceiling over the default")
ax.plot(kcurve.K, -kcurve.override_both, "-s", color=C_OOF, label="average cost of top-2-by-CV")
ax.plot(kcurve.K, kcurve.override_one.abs(), "-^", color=C_HEDGE, label="one id vs default (absolute)")
ax.set_xticks([5, 10, 15, 20, 30])
ax.set_xlabel("candidates in the pool  (= submissions your agent actually made)")
ax.set_ylabel("private AUC")
ax.legend(fontsize=10)
ax.set_title("Fewer submissions shrink the whole selection question.\n"
             "The ordering of the policies never flips.", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

> **Takeaway.** Everything shrinks with the budget. At five submissions the entire hindsight
> ceiling is +0.00007, and the average damage of top-2-by-CV falls to a sixth of its
> thirty-candidate size, simply because two picks out of five leave little room to differ.
> The ordering never flips: one id stays within noise of the default at every K. If your
> agent submits a handful of times, selection deserves even less of its hour, in both
> directions.

<a id="8"></a>
## 8. Selection is nearly solved. The score is somewhere else

Compare everything selection can buy against what candidate quality can.

In [ ]:
spread = pd.DataFrame([dict(dataset=n, worst=p.private.min(),
                            median=p.private.median(), best=p.private.max())
                       for n, p in pools.items()]).set_index("dataset")
spread["from_worst"] = spread.best - spread.worst
spread["from_median"] = spread.best - spread["median"]

head_vals = (two.oracle - two.harness_default).values
head = head_vals.mean(); hlo, hhi = boot_ci(head_vals)
print(f"headroom above the harness default, with hindsight : {head:.5f}  "
      f"90% CI [{hlo:.5f}, {hhi:.5f}]")
print(f"best candidate minus worst in the pool             : {spread.from_worst.mean():.5f}")
print(f"best candidate minus MEDIAN in the pool            : {spread.from_median.mean():.5f}")
print(f"\nmodel quality is worth this much more than selection:")
print(f"  against the worst candidate : {spread.from_worst.mean()/hhi:.0f}x to "
      f"{spread.from_worst.mean()/hlo:.0f}x")
print(f"  against a median candidate  : {spread.from_median.mean()/hhi:.0f}x to "
      f"{spread.from_median.mean()/hlo:.0f}x")
show(spread.sort_values("from_median", ascending=False).head(5), precision=4)

Selection headroom and model quality on one axis, which is the only fair way to compare them:

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 3.9))
order = spread.sort_values("from_median"); y = np.arange(len(order))
ax.hlines(y, order["median"], order.best, color=C_GREY, lw=3, zorder=2)
ax.scatter(order.best, y, s=45, color=C_ORACLE, zorder=3, label="best candidate in the pool")
ax.scatter(order["median"], y, s=45, color=C_OOF, zorder=3, label="median candidate")
ax.set_yticks(y)
ax.set_yticklabels([d.replace("train_", "") for d in order.index], fontsize=10)
ax.set_xlabel("private AUC")
ax.set_ylabel("dataset (train_NN)")
ax.legend(fontsize=10, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2,
          frameon=False)
ax.set_title(f"Grey bar: what a better model wins. Selection's entire prize is "
             f"{head:.5f},\nwhich at this scale is narrower than one of these markers.",
             fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()

One distinction before the caveats, because the title of this notebook depends on it.

**That 0.00049 is an upside, and only an upside.** It is the most a perfect oracle could add on
top of the harness default. It is not a bound on what selection can *cost* you: section 7's
worst policy gave up 0.01664 on train_13, which is more than thirty times the entire upside.
Selection here is a game with a very small prize and a real forfeit, and that asymmetry is why
the advice in section 9 is about protecting the downside rather than pursuing the gain. Any
effort you spend trying to *win* selection is chasing five ten-thousandths of AUC.

One caveat on the ratio itself: "best minus worst" flatters me, because the pool deliberately
contains a two-leaf stump no agent would submit. That is why the output above also reports the
distance from the *median* candidate, which is the fairer comparison and still large.

> **Takeaway.** The whole selection question, played perfectly with full knowledge of the private
> half, is worth about five ten-thousandths of AUC. Moving from a median candidate to the best
> one in the same pool is worth fifteen to seventy times that, and the comparison against the
> genuinely bad candidates runs to several hundred. Selection is a rounding error; the model is
> not. If your agent has an hour, this is where the hour goes.

### The one other published estimate, and it agrees

Someone else has priced this. Krizsó Gergely's
[AgentForge Observatory](https://www.kaggle.com/code/lucifer19/agentforge-observatory) treats the final choice as a decision problem:
candidates arrive in near-duplicate clusters, the maximum of K noisy public scores is upward
biased, and averaging inside a cluster before shrinking should therefore choose better than the
raw top two. Its Monte Carlo puts the gain over that same top-two-public baseline at **+0.0001
to +0.0005** private AUC, and reads its own result strictly: the gain *"sits at or below the
promotion margin"*, so the rule ships only as a superset of the anchor rule rather than on its
own merits.

That is a closely related quantity, not the same one, and the difference matters. His figure is
what one rule returns against the top-two-public baseline; the 0.00049 above is the most *any*
rule could return against that baseline, with perfect hindsight. A rule landing at or below the
ceiling is arithmetic, not corroboration. What is informative is the size: his realised gain sits
at the same order of magnitude as my ceiling rather than ten times under it, which says the
ceiling is close to reachable and that we are both looking at a very small number. His number
comes
from a model of the noise with no real data in it; mine comes from sixteen real pools with the
answers attached and no model of anything. The point estimate here, 0.00049, lands inside his
+0.0001 to +0.0005 band; my interval runs a little wider than his at the top. Two estimates that
share no assumptions landing in the same neighbourhood is worth more than either alone.

His design note also gets to the algebra first, so it belongs to him and not to me: shrinking
every candidate's public score toward the pool mean by a common factor is an increasing linear
map, and a single global factor *"would be order preserving and could not change any decision"*.
That is precisely why his rule shrinks per cluster instead of globally.

Which leaves the family I *can* test on this data. Shrinkage moves a pick only by pulling toward
a second signal, so the family is every convex blend of the two rulers, with the blending weight
tuned on the private half itself, a licence no real agent has.

In [ ]:
def _z(v):
    return (v - v.mean()) / (v.std() + 1e-12)

same = sum(int(p.public.values.argmax() ==
               (p.public.mean() + 0.9 * (p.public.values - p.public.mean())).argmax())
           for p in pools.values())
print(f"pure shrinkage of public scores picks the same candidate on {same} of {len(pools)} "
      f"datasets, as the algebra says it must\n")

blend = []
for w in np.linspace(0, 1, 21):
    picks = [p.private.values[(w * _z(p.public.values) + (1 - w) * _z(p.oof.values)).argmax()]
             for p in pools.values()]
    blend.append(dict(weight_on_public=round(w, 2), private=np.mean(picks)))
blend = pd.DataFrame(blend)
best = blend.loc[blend.private.idxmax()]
default = two.harness_default.mean()
print(f"best blend of the two rulers : {best.private:.5f}  (at weight {best.weight_on_public:.2f} "
      f"on the public half)")
print(f"the harness default          : {default:.5f}")
print(f"difference                   : {best.private - default:+.5f}")
print(f"\nand that blend weight was chosen by looking at the private half, which no agent can do")

# the cluster rule needs near-duplicate candidates to work on; count them in this pool
from scipy.stats import rankdata
pair_counts = []
for name, k in kept.items():
    R = rankdata(k["preds"], axis=1)
    C = np.corrcoef(R)
    pair_counts.append(sum(1 for i in range(len(C)) for j in range(i + 1, len(C))
                           if C[i, j] >= 0.999))
pair_counts = np.array(pair_counts)
print(f"\npools with no candidate pair correlating at Spearman >= 0.999: "
      f"{int((pair_counts == 0).sum())} of {len(pair_counts)}")
print(f"most such pairs in any one pool: {pair_counts.max()}")

> **Takeaway.** The best member of that whole family, with its weight tuned against the answers,
> lands **0.00002 below** simply taking the two best public scores. Not below the oracle: below
> doing nothing. Which is the sharper version of the headroom number: even the 0.00049 that
> hindsight leaves on the table is not reachable by reweighting the two signals an agent can
> actually see.
>
> One thing that is *not* tested above. The cluster rule pulls on a signal my pool barely has.
> It groups candidates whose predictions rank alike at Spearman 0.999 and averages inside the
> group, which needs rank blends of shared sources to bite. My thirty candidates are thirty
> distinct hyperparameter settings, and the cell above counts what that leaves: eleven of the
> sixteen pools contain no such pair at all, and no pool contains more than three. So this
> notebook prices the ceiling and his prices a rule for approaching it, in a pool built
> differently from mine. Neither of us finds much there, which is the point both of us end up
> making.

### What the public half can and cannot see

A prize of five ten-thousandths only means something next to some scale of comparison. The one
that fits is the public half's own resolution: how finely can it tell two candidates apart at all?

Take every pair of candidates that lands within a whisker of the same public AUC, and measure how
far apart the private half puts them.

In [ ]:
import itertools

rows = []
for tol in (0.0002, 0.0010, 0.0030):
    d_private, d_full = [], []
    for name, p in pools.items():
        pub, prv = p.public.values, p.private.values
        k = kept[name]; y = k["truth"]
        # AUC on all 10000 test rows: a number no Public/Private assignment can change
        full = np.array([roc_auc_score(y, k["preds"][i]) for i in range(len(p))])
        for i, j in itertools.combinations(range(len(p)), 2):
            if abs(pub[i] - pub[j]) <= tol:
                d_private.append(abs(prv[i] - prv[j]))
                d_full.append(abs(full[i] - full[j]))
    d_private, d_full = np.array(d_private), np.array(d_full)
    rows.append(dict(tie_tolerance=tol, pairs=len(d_private),
                     median_private_gap=np.median(d_private),
                     pct90=np.percentile(d_private, 90),
                     median_full_set_gap=np.median(d_full),
                     survives_every_split=np.median(d_full) / np.median(d_private)))
tied = pd.DataFrame(rows).set_index("tie_tolerance")
show(tied, precision=5)
print(f"\nselection's entire oracle prize, from the cell above: {head:.5f}")

The tempting reading of the middle row is that 0.00100 is the noise in the split, and that
anything smaller is unmeasurable. That reading is wrong, and the last two columns are the control
that shows it.

Score the same candidates on all ten thousand test rows. That number does not depend on where the
Public/Private line falls, so nothing in it can be split noise. Among pairs the public half calls
tied to 0.001, the typical pair is still 0.00061 apart on the full set: **61% of the apparent
noise is real model quality that the public half could not resolve.** Loosen the tie to 0.003 and
the share climbs to 86%, since a looser tie lets in candidates that genuinely differ. Tighten it
to 0.0002 and it falls to 48%.

So the public half is not mostly lying to you. It is mostly running out of resolution.

> **Takeaway.** Two candidates that the public half cannot tell apart sit a median 0.00100 of
> private AUC apart, and one pair in ten sits more than 0.00291 apart. Selection's entire prize,
> played perfectly with hindsight, is 0.00049. The prize is smaller than the gap between two
> candidates your ruler calls a tie.
>
> That is why section 7 is about protecting the downside instead of chasing the gain. Your agent
> is picking inside a band its instrument cannot read, and the only move there with a measurable
> consequence is the one that loses 0.01664.

<a id="9"></a>
## 9. What this means for your agent

### What published agents actually do

I read the selection instruction in twelve published agents. Selection is already being reasoned
about, carefully, and not the way I concluded. Krizsó Gergely gives it its own stage: his
[AgentForge](https://www.kaggle.com/code/lucifer19/agentforge-autonomous-binary-classifier) hands the last step to a sub-agent named
`final_selection_controller` whose only tools are `get_status` and `select_submission`, and the
companion [Observatory](https://www.kaggle.com/code/lucifer19/agentforge-observatory) lists *"Selection is a decision problem"* among its
design principles.
navazshfathi's [agent](https://www.kaggle.com/code/navazshfathi/autonomous-agent-prediction-beta)
runs a deliberate public-versus-CV hedge across the two slots. And Busya PRIME's
[metric-aware agent](https://www.kaggle.com/code/busyaprime/a-metric-aware-autonomous-ml-agent)
comes closest to the question in one line: *"Select your best out of fold model as final, not
your best public one."* Note the singular. One model, chosen by cross-validation, is option B
below rather than the two-id policy section 7 finds expensive. The disagreement is narrower than
that line makes it sound: sections 5 and 6 find the public half the better ruler for picking that
one id on the six datasets smaller than it. Plenty of people thought about this before I did.

**Up to the day this notebook went up, nobody left a slot to the harness as a selection
decision.** Two published agents did pass a single id.
SIDHAARTH SHREE's [A-to-Z Guide](https://www.kaggle.com/code/sidhaarthshree/autonomous-agent-prediction-a-to-z-guide) ends its workflow *"Call get_status, then
select_submission with your best submission ID"*, and Nur Srijan's
[starter](https://www.kaggle.com/code/nursrijan/agent-starter-dynamic-automl-guide) says *"select the best submission ID for final scoring"*. Neither
says why one rather than two, and neither mentions the slot it leaves behind, so both land on
the policy of section 7 without arguing for it. Everyone else claimed both slots. The widely
shared prompt tree, which AgentForge and Naji's earlier
[template](https://www.kaggle.com/code/najiama/lb-0-822-easy-to-edit-agent-template) both carried,
instructs: *"If two or more successful submissions exist, call `select_submission` with the two
distinct IDs having the highest public scores."* Read against section 2, that is a careful
hand-built version of what the harness already does by default. The policy is right.

**Postscript, 2 August 2026.** The field moved while the competition was still running. Naji's [freeroll notebook](https://www.kaggle.com/code/najiama/lb-0-823-the-freeroll-gemini-pro-strategy)
replaced its selection agent with a "No-Select Architecture": the stage is deleted outright, the
harness fills both slots, its stage prompts now *forbid* calling `select_submission`, and the
change is credited to the measurement in this notebook. What section 7 priced as "doing nothing
is equally fine" became, within three days, the architecture of the competition's second
most-voted community notebook — a better test of the numbers than any argument I could add here.

**Postscript, after the deadline.** The first write-up to touch the question puts it directly. Nikhil
Krishna A's [write-up](https://www.kaggle.com/competitions/autonomous-agent-prediction-beta/discussion/733564) reports Public 0.816 and Private
0.779 across nine submitted versions, and writes under *What I'd Do Differently*: *"Fix
`select_submission` as a hard invariant. Every version of the agent should have
`select_submission` guaranteed — it should be the last action regardless of what else goes
wrong."* Section 2 narrows what that guarantee buys. The failure it insures against does not
exist: with submissions on file, never calling the tool is a complete policy, not a zero. What
the guarantee does control is which ids get passed, and section 7 is where that matters — a
guaranteed call carrying two cross-validation picks is the one move measured here that can cost
you something.

He gives the reason as a Key Lesson: *"Reliability first. An agent that errors or fails to call
`select_submission` scores zero."* The first half is right, and it is the larger risk by far: an
agent that dies mid-session has nothing on file, and `get_final_submissions` returns an empty
list. The second half is the belief section 2 was written to check.

His other regret is the one section 6 measured. He wanted a loop he never got to run — *"Our
prescriptive prompts prevented this"* — and describes it as *"Build a submit → observe → improve
loop. The agent sees public scores mid-session. Design the prompt to exploit this: submit a
baseline, read the score, try an improvement, resubmit."* The chase study prices that loop at
+0.0136 private AUC against +0.0060 for the same budget spent without feedback, and finds no
extra leaderboard optimism for the trouble. The regret was worth having.

> **Takeaway.** The auto-fill got one passing sentence on the competition page, a singular
> (and narrower) description in the docs the agent reads, and its one-id partial form appeared
> only in the source. Of its two readings, the field took one and left the other. The
> *crash-insurance* reading went mainstream inside three days:
> [Naji's freeroll](https://www.kaggle.com/competitions/autonomous-agent-prediction-beta/discussion/730539)
> banks a cheap baseline and lets an expensive model gamble the rest of the hour behind the
> fallback. The *selection-policy* reading stayed unused for the life of the competition: two
> agents passed one id without arguing for it, and none reasoned about the slot it left behind.
> If this harness comes back, that is where the free option is.

### So, concretely

**Claiming one slot instead of two is the cautious option.** It leaves your best public score in
the slot you did not claim, so your agent's judgment gets a free shot while the fallback stays
in play. Against doing nothing it measured as a wash; its worst case is bounded in a way that
claiming both slots is not.

**Doing nothing is equally fine.** If you prefer to be explicit, pass your two best *public* ids:
identical outcome, easier to read in a trace.

**Delete "select your best two submissions" from your prompt.** In the two-slot form that is the
only policy that lost badly anywhere in section 7, and it lost for the reason section 6 gives.

**The host's own demo prompt disagrees with me.** Ryan Holbrook's
[demo agent](https://www.kaggle.com/code/ryanholbrook/autonomous-agent-prediction-beta-demo-agent)
tells the model to *"avoid overfitting to public leaderboard scores"*. That is correct Kaggle
advice in general; sections 5 and 6 measure whether it holds here, and the thirty-round chase
at the end of section 6 prices it directly: the chaser picked up +0.002 of public-score
optimism, the same as not chasing, and twice the private improvement. Take the measurement,
not my confidence.

**Then go build candidates.** Section 8 is the one I would keep if I could keep only one.

### The lines themselves

Two configurations, both defensible. Pick one and stop thinking about it.

**A. Take the default.** Leave `select_submission` out of the tool list entirely, the way the
host's reference agent does. Your agent physically cannot spend the slots, and the harness
gives you the best two public scores.

```yaml
# agent.yaml
tools:
  - submit_predictions
  # select_submission deliberately omitted: the harness fills both slots
  # with the best public scores, which is the policy I want anyway.
```

**B. Claim one slot, keep the fallback.** Add the tool, and be explicit in the prompt that it
takes one id. The wording matters, because "select your best submissions" reliably produces two.

```yaml
# agent.yaml
tools:
  - submit_predictions
  - select_submission
```

```markdown
<!-- prompts/system.md -->
Before you finish, call select_submission with EXACTLY ONE id: your single best
model by cross-validation. Pass one id, not two. The harness automatically fills
the remaining slot with your best public score, and that fallback is worth keeping.
```

**And one line to delete, wherever it appears in your prompt.** Anything of the form "select
your best two submissions" or "lock in your top 2".

<a id="10"></a>
## 10. Design choices & limitations

- **`pd.read_csv` segfaults on one of these files, and your agent should know.** On the Kaggle
  image, reading `data/train_14/test.csv` with pandas 2.3.3 kills the process outright: exit
  code -11, no traceback, with the C engine, the python engine and chunked reads alike. I lost
  three notebook runs to it before instrumenting per-file. `engine="pyarrow"` reads the same
  file to identical values and survives, so every read here goes through a two-line wrapper. If
  your agent loads its data the obvious way and the sandbox dies without an error message, this
  is a candidate.

- **There is a third split I cannot see.** `agent_tools.md` says the harness evaluates against
  Public, Private *and* Holdout, but the sixteen training packages carry only Public and
  Private in `solution.csv`. Everything above is measured on the split I can observe.
- **Sixteen datasets is a small sample**, which is why every headline comparison carries a
  resampled interval rather than a point estimate. Several of those intervals cross zero, and
  I have tried to say so where they do rather than round them off.
- **The pool matters, and I measured how much.** Every candidate above is LightGBM. Two more
  pools, run offline with the same folds and seed, check what that costs. A differently-composed
  thirty LightGBM variants kept every sign, with a smaller section 5 margin (+0.0004, interval
  crossing zero). A thirty-candidate pool spanning four families — LightGBM, XGBoost, CatBoost,
  plus logistic regressions and extra-trees — kept every verdict that matters: the median dataset
  still shows exactly zero, the public half's advantage still lives only below 5,000 training
  rows (+0.0049 there, −0.0002 above), and the section 7 asymmetry comes back *sharper*: top-2-by-CV
  still gives up 0.0129 on the 500-row dataset, while passing one id never lost to the default on
  any of the sixteen and its average gain turns sign-stable (+0.0004, 90% CI +0.0000 to +0.0008).
  What moves with the pool is magnitude: the hindsight ceiling doubles to +0.0010 and
  model-over-median-candidate shrinks to 5–42x. Direction is the finding; sizes are
  pool-dependent.
- **The chase in section 6 is a simulation, not an agent.** One seeded greedy LightGBM
  chain per dataset, random tweaks, single full-train fits. A real agent proposes smarter and
  occasionally leakier changes; read the +0.002 as the cost of this kind of chasing, not an
  upper bound for all of it. Checked offline across three independently seeded chains: the
  chaser's optimism stayed within ±0.001 of the control's in every chain (and once came out
  *below* it), while its private gain stayed roughly double, +0.014 to +0.015 against +0.006
  to +0.008.
- **5-fold cross-validation**, which favours the losing side. A real agent training thirty
  candidates in an hour would run fewer folds, and its out-of-fold estimates would be noisier
  than these. Richer validation still cannot create training rows on the small datasets, which
  is where the gap lives.
- **One seed.** Fold splits and LightGBM both use seed 0. The intervals cover uncertainty about
  which datasets the family hands you, not uncertainty from retraining.
- **The source is the shipped harness, not the graded one.** Section 2 reads `kaggle_kaggle`
  version 0.1.0, the wheel packaged with the competition data on 2026-07-06 and the thing
  `run_local_eval.py` runs. Official scoring happens on Kaggle's infrastructure, which I cannot
  inspect, so I am describing the reference implementation you test against. If the wheel is
  updated, re-run section 2 before trusting section 9.
- **Nothing here was confirmed on the graded harness.** Two limits compound, and they are worth
  reading together. The source above is `kaggle_kaggle` 0.1.0 as shipped with the data, not the
  scorer Kaggle runs; and every submission I made died before the agent's first step with a 403
  from the model proxy (`User location "RU" is not supported for this model/API`, three models,
  two sampling configs), so I finished the competition without a score. The auto-fill is
  therefore read from the reference implementation and never once observed in the harness that
  actually grades. A graded run would not have settled the comparison either — it returns one
  AUC on one hidden dataset — but it would have confirmed the mechanism, and I cannot claim
  that it did.
- **This measures selection, not agents.** No LLM was involved in any number above. A real
  agent also has to write working code inside a budget, and none of that is tested here.
- **Reproducibility.** No internet, standard Kaggle CPU image, fixed seed throughout.

<a id="11"></a>
## 11. Further reading

**Start here for mechanics.**
[Demo Agent](https://www.kaggle.com/code/ryanholbrook/autonomous-agent-prediction-beta-demo-agent)
and [AIDE Agent](https://www.kaggle.com/code/ryanholbrook/autonomous-agent-prediction-beta-aide-agent)
by Ryan Holbrook. The first is the host's own starting point, the second a research-style
agent loop. Read them before writing your own `agent.yaml`.

**The community agent that thinks hardest about selection.**
[AgentForge: Autonomous Binary Classifier](https://www.kaggle.com/code/lucifer19/agentforge-autonomous-binary-classifier)
by Krizsó Gergely, and its companion [Observatory](https://www.kaggle.com/code/lucifer19/agentforge-observatory), which is where the
selection study lives. Between them they are the only entry I found that puts an actual figure
on what selection is worth; section 8 checks that figure against mine. Read the staged-pipeline
design regardless.

**The first write-ups.** Two landed on 7 August 2026, the day after the deadline.
[Nikhil Krishna A](https://www.kaggle.com/competitions/autonomous-agent-prediction-beta/discussion/733564) is the one to read on reliability: nine
submitted versions at Public 0.816 / Private 0.779, a model switch he calls *"the decisive fix"*,
and a lessons list that opens on reliability before it gets to CatBoost and multi-seed
ensembling.
[Jeki Wan Taufik](https://www.kaggle.com/competitions/autonomous-agent-prediction-beta/discussion/733628) is the one to read on structure: Public 0.826 /
Private 0.779 from a main orchestration agent plus a specialised analysis agent, built *"to adapt its workflow to unseen
tabular datasets through task decomposition and reusable prompts"* rather than hard-coding
per-dataset logic. Read them for what this notebook deliberately does not cover, which is
everything about getting an agent through its hour alive.

**A worked multi-agent architecture.**
[Meta-Learning Pipeline for Sandboxed AutoML](https://www.kaggle.com/code/avikdas567/meta-learning-pipeline-for-sandboxed-automl)
by Avik Das, for how to wire several specialised agents together under the budget.

**On the budget.**
[Is LLM budget only \$2?](https://www.kaggle.com/competitions/autonomous-agent-prediction-beta/discussion/723806),
started by Chris Deotte, where the host confirms the cap is per session and *"\$4 per
submission"*. Running it yourself is a different bill: in
[a separate thread](https://www.kaggle.com/competitions/autonomous-agent-prediction-beta/discussion/724120) the host answers that you will need *"an API key to
some model provider"* plus a container runtime. The official runs do not use your key; they route
through the Kaggle model proxy that the shipped `.env.example` describes.

**Credits.** The format, the sixteen training datasets and the local harness are by Ryan Holbrook and
Addison Howard. Two things they shipped made this notebook possible: the `Usage` column in
`solution.csv`, and the harness source in the wheels.

If you disagree with section 7, the code is above and the datasets are the ones you already
have. I would rather be corrected with a re-run than agreed with.